In [26]:
import requests
import pandas as pd
import numpy as np
import time
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pathlib import Path

In [27]:
API_KEY = "PKYH3LLZJT2YLH6OXTFVGSOUUK"
SECRET_KEY = "BuN7pWZjoB62RjsWvuQ9gV6TZHMFdLhvPQFZ27JRnekQ"

HEADERS = {
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": SECRET_KEY,
}

BASE_URL = "https://data.alpaca.markets/v1beta1/news"
CACHE_DIR = Path("alpaca_news_cache")
CACHE_DIR.mkdir(exist_ok=True)

In [28]:
stock_returns = pd.read_csv("stock_returns.csv")
etf_returns = pd.read_csv("etf_returns.csv")

stock_returns["date"] = pd.to_datetime(stock_returns["date"])
stock_returns = stock_returns.set_index("date").sort_index()

etf_returns["date"] = pd.to_datetime(etf_returns["date"])
etf_returns = etf_returns.set_index("date").sort_index()

tickers = stock_returns.columns.tolist() + etf_returns.columns.tolist()
tickers

['AAPL',
 'AMZN',
 'CAT',
 'JNJ',
 'JPM',
 'KO',
 'MSFT',
 'NVDA',
 'V',
 'XOM',
 'EEM',
 'EFA',
 'IWM',
 'QQQ',
 'SPY',
 'TLT',
 'VNQ',
 'XLE',
 'XLK',
 'XLV']

In [29]:
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

In [30]:
def fetch_news_batch(symbols, start, end, limit=50, sleep_sec=0.2):
    rows = []
    page_token = None

    while True:
        params = {
            "symbols": ",".join(symbols),
            "start": start,
            "end": end,
            "limit": limit,
            "sort": "asc"
        }
        if page_token:
            params["page_token"] = page_token

        r = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        payload = r.json()

        news_items = payload.get("news", [])
        for item in news_items:
            rows.append({
                "id": item.get("id"),
                "headline": item.get("headline"),
                "summary": item.get("summary"),
                "author": item.get("author"),
                "source": item.get("source"),
                "url": item.get("url"),
                "created_at": item.get("created_at"),
                "updated_at": item.get("updated_at"),
                "symbols": item.get("symbols"),
            })

        page_token = payload.get("next_page_token")
        if not page_token:
            break

        time.sleep(sleep_sec)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
        df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce", utc=True)

    return df

In [31]:
def get_month_news_cached(symbol_batch, month_end):
    month_start = month_end.to_period("M").start_time
    start = month_start.strftime("%Y-%m-%dT00:00:00Z")
    end = month_end.strftime("%Y-%m-%dT23:59:59Z")

    batch_name = "_".join(symbol_batch)
    cache_file = CACHE_DIR / f"{month_start.strftime('%Y_%m')}_{batch_name}.parquet"

    if cache_file.exists():
        return pd.read_parquet(cache_file)

    df = fetch_news_batch(symbol_batch, start, end, limit=50, sleep_sec=0.25)
    df.to_parquet(cache_file, index=False)
    return df

In [ ]:
month_ends = pd.date_range("2015-01-01", "2025-12-31", freq="M")

all_news = []

for month_end in month_ends:
    print("Month:", month_end.strftime("%Y-%m"))
    for batch in chunk_list(tickers, 5):
        try:
            df_batch = get_month_news_cached(batch, month_end)
            if not df_batch.empty:
                all_news.append(df_batch)
        except Exception as e:
            print("Failed:", month_end.strftime("%Y-%m"), batch, e)

news_df = pd.concat(all_news, ignore_index=True) if all_news else pd.DataFrame()
news_df.head()

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_42536/2516320819.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_ends = pd.date_range("2015-01-01", "2025-12-31", freq="M")


Month: 2015-01
Failed: 2015-01 ['AAPL', 'AMZN', 'CAT', 'JNJ', 'JPM'] Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
Failed: 2015-01 ['KO', 'MSFT', 'NVDA', 'V', 'XOM'] Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or co